# NB_lab_test_trends

Builds `<catalog>.gold.lab_test_trends` (one row per test_name per calendar month) from silver lab_results, with month/quarter names from the calendar and month-over-month abnormal rate trend via LAG(). Full rebuild via CREATE OR REPLACE TABLE ... AS SELECT.

In [0]:
# Catalog placeholder from the STTM (<catalog>)
dbutils.widgets.text("catalog", "databricks_caresync")
catalog = dbutils.widgets.get("catalog")

source_schema = f"{catalog}.silver"
target_table = f"{catalog}.gold.lab_test_trends"
print(target_table)

In [0]:
# Driving table = silver.lab_results grouped by test_name, year, month of test_date
# Note: STTM references calendar.date_key; the silver calendar column is datekey (INT yyyyMMdd), joined on the first day of the month
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table}
COMMENT 'Gold: one row per lab test type per month with abnormal/critical rates and month-over-month trend'
AS
WITH monthly AS (
  SELECT
    test_name,
    year(test_date)                                         AS year,
    month(test_date)                                        AS month,
    CAST(date_trunc('MONTH', test_date) AS DATE)            AS month_start_date,
    COUNT(*)                                                AS total_tests,
    COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END)  AS abnormal_test_count,
    COUNT(CASE WHEN result_status = 'Critical' THEN 1 END)  AS critical_test_count
  FROM {source_schema}.lab_results
  GROUP BY test_name, year(test_date), month(test_date), CAST(date_trunc('MONTH', test_date) AS DATE)
),
rates AS (
  SELECT
    m.*,
    CAST(ROUND(try_divide(abnormal_test_count, total_tests) * 100, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(ROUND(try_divide(critical_test_count, total_tests) * 100, 2) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
with_lag AS (
  SELECT
    r.*,
    LAG(r.abnormal_rate_pct) OVER (PARTITION BY r.test_name ORDER BY r.year, r.month) AS prev_month_abnormal_rate_pct
  FROM rates r
)
SELECT
  CAST(w.test_name AS STRING)                                AS test_name,
  CAST(w.year AS INT)                                        AS year,
  CAST(w.month AS INT)                                       AS month,
  CAST(c.month_name AS STRING)                               AS month_name,
  CAST(c.quarter_name AS STRING)                             AS quarter_name,
  CAST(w.total_tests AS INT)                                 AS total_tests,
  CAST(w.abnormal_test_count AS INT)                         AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                         AS critical_test_count,
  w.abnormal_rate_pct                                        AS abnormal_rate_pct,
  w.critical_rate_pct                                        AS critical_rate_pct,
  CAST(w.prev_month_abnormal_rate_pct AS DECIMAL(5,2))       AS prev_month_abnormal_rate_pct,
  CAST(w.abnormal_rate_pct - w.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM with_lag w
LEFT JOIN {source_schema}.calendar c
  ON c.datekey = CAST(date_format(w.month_start_date, 'yyyyMMdd') AS INT)
""")

display(spark.table(target_table).orderBy("test_name", "year", "month"))

In [0]:
print("=" * 80)
print("[STEP 5] Computing processed record count")
print("=" * 80)

# Gold table rebuilt in the previous cell
updated_df = spark.table(target_table)
record_count = updated_df.count()

print(f"[STEP 5 COMPLETE] Processed record count: {record_count}")

dbutils.notebook.exit(str(record_count))